In [ ]:
import requests, time, json, os
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

SITE = "dream2000"
BASE = "https://dream2000.com"
CONTACT_EMAIL = "ayaelgrwany18@gmail.com"
HEADERS = {"User-Agent": f"DepiPriceBot/0.1 (student project; +mailto:{CONTACT_EMAIL})"}

CATEGORIES = {
    "phone": ["mobiles"],
    "tv": ["brands", "lg-tvs", "lg-tv", "hisense-tvs", "haier-tvs", "haier-tv",
           "fresh-tvs", "fresh-tv", "ata-tvs", "armadillo-tvs", "master-electron-tvs"],
}

snapshot_date = datetime.now(ZoneInfo("Africa/Cairo")).strftime("%Y-%m-%d")
print("snapshot_date:", snapshot_date)

snapshot_date: 2026-10-29


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
WORK_DIR = "/content/drive/MyDrive/depi_scraping"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print("بنحفظ في:", os.getcwd())

Mounted at /content/drive
بنحفظ في: /content/drive/MyDrive/depi_scraping


In [ ]:
os.makedirs(f"raw/{snapshot_date}/{SITE}", exist_ok=True)

def fetch_collection(handle):
    products = []
    page = 1
    while True:
        url = f"{BASE}/collections/{handle}/products.json?limit=250&page={page}"
        r = requests.get(url, headers=HEADERS, timeout=30)
        time.sleep(1)
        if r.status_code != 200:
            print(handle, "page", page, "status", r.status_code)
            break
        batch = r.json()["products"]
        if not batch:
            break
        raw_file = f"raw/{snapshot_date}/{SITE}/{handle}_page{page}.json"
        with open(raw_file, "w", encoding="utf-8") as f:
            f.write(r.text)
        scraped_at = datetime.now(timezone.utc)
        for p in batch:
            products.append((p, handle, raw_file, scraped_at))
        page += 1
    return products



In [ ]:
import re
import pandas as pd

def to_offers(item, category):
    p, handle, raw_file, scraped_at = item
    offers = []
    for v in p["variants"]:
        title = " ".join(str(p["title"]).split())
        if v.get("title") and v["title"] != "Default Title":
            title = f"{title} - {v['title']}"
        sku = str(v.get("sku") or "").strip()
        is_ean = bool(re.fullmatch(r"\d{13}", sku))
        price = float(v["price"]) if v.get("price") else None
        compare = float(v["compare_at_price"]) if v.get("compare_at_price") else None
        original = compare if (compare and price and compare > price) else None
        offers.append({
            "site": SITE,
            "site_product_id": f"{p['handle']}:{v['id']}",
            "snapshot_date": snapshot_date,
            "scraped_at": scraped_at,
            "url": f"{BASE}/products/{p['handle']}?variant={v['id']}",
            "title": title,
            "brand_raw": p.get("vendor"),
            "model_raw": sku if (sku and not is_ean) else None,
            "ean": sku if is_ean else None,
            "category_site": handle,
            "category": category,
            "sale_price": price,
            "original_price": original,
            "discount_pct": round(100 * (1 - price / original)) if original else None,
            "currency": "EGP",
            "stock_text": None,
            "in_stock": v.get("available"),
            "seller": None,
            "source_format": "json",
            "raw_file": raw_file,
        })
    return offers



In [ ]:
all_offers = []
for category, handles in CATEGORIES.items():
    for h in handles:
        items = fetch_collection(h)
        for item in items:
            all_offers.extend(to_offers(item, category))
        print(category, h, len(items), "منتج")

df = pd.DataFrame(all_offers)
print("قبل حذف المكرر:", len(df))
df = df.drop_duplicates(subset=["site", "site_product_id", "snapshot_date"])
print("بعد حذف المكرر:", len(df))
print(df["category"].value_counts())

phone mobiles 117 منتج
tv brands 20 منتج
tv lg-tvs 11 منتج
tv lg-tv 11 منتج
tv hisense-tvs 5 منتج
tv haier-tvs 11 منتج
tv haier-tv 10 منتج
tv fresh-tvs 2 منتج
tv fresh-tv 2 منتج
tv ata-tvs 9 منتج
tv armadillo-tvs 6 منتج
tv master-electron-tvs 0 منتج
قبل حذف المكرر: 618
بعد حذف المكرر: 583
category
phone    523
tv        60
Name: count, dtype: int64


In [ ]:
df["reason"] = None
df.loc[df["in_stock"].isna(), "reason"] = "no_stock_info"
df.loc[df["sale_price"].isna() | (df["sale_price"] <= 0), "reason"] = "no_sale_price"

quarantine = df[df["reason"].notna()]
offers_ok = df[df["reason"].isna()].drop(columns="reason")
print("سليم:", len(offers_ok), "| quarantine:", len(quarantine))

name = f"{SITE}_offers_{snapshot_date}"
offers_ok.to_json(f"{name}.json", orient="records", force_ascii=False, date_format="iso", indent=1)
quarantine.to_json(f"{name}_quarantine.json", orient="records", force_ascii=False, date_format="iso", indent=1)
offers_ok.to_csv(f"{name}.csv", index=False, encoding="utf-8-sig")
print("اتحفظ:", name)

سليم: 583 | quarantine: 0
اتحفظ: dream2000_offers_2026-10-29


In [ ]:
quarantine[["title", "sale_price", "in_stock", "reason", "url"]]

,title,sale_price,in_stock,reason,url
